# 10. Full data: evaluation of the feature sets that helped in the pilots

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.pilots import climate, clustering, data, earth_engine, evaluation, features, full
from experiments.tuning import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
AREA = "pilots"


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)


def repeated(name, X, rows, ok, seeds=(1, 2, 3), params=evaluation.PARAMS):
    """Mean and SD over random-forest seeds of every evaluation view."""
    summaries, details = [], []
    for seed in seeds:
        summary, holdout = evaluation.evaluate(name, X, rows, ok, params, seed)
        summaries.append({**summary, "seed": seed})
        details.extend({**r, "seed": seed} for r in holdout)
    return summaries, details


def summarize(summaries):
    table = pd.DataFrame(summaries)
    metrics = ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1",
               "jecam_carrot_f1", "jecam_cabbage_f1", "jecam_potato_f1"]
    grouped = table.groupby("feature_set", sort=False)
    out = grouped[metrics].agg(["mean", "std"]).round(3)
    out.insert(0, ("pixels_used", ""), grouped["pixels_used"].first())
    return out

## Inputs

In [ ]:
full_run = ""  # Optional pipeline_outputs/pilots/subset_<id>; blank uses the latest
full_run = Path(full_run) if full_run else tuning.latest_run(root, "full_data", area=AREA)
rows, arrays = data.load(full_run)
cube = features.stack(arrays["s2_values"])
valid = arrays["s2_valid"]
print("Full data:", full_run, "| pixels:", len(rows))

In [ ]:
s2, s1 = arrays["s2_values"], arrays["s1_values"]
seeds = (1, 2)
built = {name: build(s2, valid, s1) for name, build in full.SETS.items()}
fixed = features.fixed_interval(cube, valid)
common = np.logical_and.reduce([ok for _, ok, _ in built.values()])
display(pd.DataFrame([{"feature_set": "fixed_may_aug", "predictors": fixed[0].shape[1], "pixels": int(fixed[1].sum())}] +
                     [{"feature_set": k, "predictors": X.shape[1], "pixels": int(ok.sum())} for k, (X, ok, _) in built.items()]))
print("Pixels usable by every harmonic/radar set:", int(common.sum()), "of", len(rows))

## Evaluate

In [ ]:
run = tuning.new_run(root, "full_evaluation", full_run / "meta.json", area=AREA, seeds=list(seeds))
summaries, details = [], []
s, d = repeated("fixed_may_aug (own pixels)", fixed[0], rows, fixed[1], seeds)
summaries += s
details += d
print("fixed done", flush=True)
for name, (X, _, _) in built.items():
    s, d = repeated(name, np.nan_to_num(X), rows, common, seeds)
    summaries += s
    details += d
    print(name, "done", flush=True)
for name in ("s2_harmonic", "s2_harmonic_plus_s1_ratios"):
    augmented = full.copies(full.SETS[name], s2, valid, s1)
    for seed in seeds:
        summary, holdout = evaluation.evaluate(f"{name} + augmentation", np.nan_to_num(built[name][0]), rows, common,
                                               seed=seed, copies=augmented)
        summaries.append({**summary, "seed": seed})
        details += [{**h, "seed": seed} for h in holdout]
    print(name, "+ augmentation done", flush=True)
tuning.save_records(run, "summary", summaries)
tuning.save_records(run, "dataset_holdout", details)
display(summarize(summaries))

## Spatial forward feature selection

In [ ]:
X, _, columns = built["s2_harmonic_plus_s1_raw_and_ratios"]
groups = features.column_groups(columns)
fast = {**evaluation.PARAMS, "n_estimators": 100}
selected, history = evaluation.forward_group_selection(np.nan_to_num(X), rows, common, groups, fast, seed=1,
                                                       progress=lambda m: print(m, flush=True))
tuning.save_records(run, "selection_history", history)
(run / "selected_groups.json").write_text(json.dumps(selected))
chosen = [c for g in selected for c in groups[g]]
s, d = repeated("selected_groups", np.nan_to_num(X[:, chosen]), rows, common, seeds)
summaries += s
details += d
tuning.save_records(run, "summary", summaries)
display(pd.DataFrame(history).round(3))
display(summarize(summaries))

## Dataset holdout details

In [ ]:
display(pd.DataFrame([d for d in details if d["status"] == "scored"]).groupby(["feature_set", "held_out_source"])
        [["macro_f1_scored_crops", "carrot_recall", "potato_recall"]].mean().round(3))

## Official pipeline results for reference

In [ ]:
from crop_pipeline.analysis_workflow import resolve_stage_pointer
scores_handoff = resolve_stage_pointer(root / "pipeline_outputs/testing/current_scores.json")
official = json.loads((scores_handoff.parent / "reports/test_scores.json").read_text())
reference = [{"model": f"official {k}", "jecam_macro_f1": v.get("macro_f1"),
              "jecam_parcels": sum(c["support"] for c in v["per_crop"].values())} for k, v in official.items() if "per_crop" in v]
latest = sorted((root / "pipeline_outputs/transfer").glob("final_check_*"), key=lambda p: p.stat().st_mtime)
if latest:
    check = pd.read_csv(latest[-1] / "scores.csv")
    reference += [{"model": f"transfer combined {r.cohort}", "jecam_macro_f1": r.macro_f1, "jecam_parcels": r.parcels}
                  for r in check[(check.stage == "jecam_test") & (check.configuration == "combined")].itertuples()]
display(pd.DataFrame(reference).round(3))

## Figure

In [ ]:
table = pd.DataFrame(summaries).groupby("feature_set", sort=False)[
    ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1"]].mean()
figure, axis = plt.subplots(figsize=(13, 4))
table.T.plot(kind="bar", ax=axis)
axis.set_ylim(0, 1)
axis.set_ylabel("Parcel macro-F1 (mean of seeds)")
axis.legend(fontsize=7)
show_and_save(figure, run, "full_evaluation")
print("Run folder:", run)